# 01 — Spooky 👻: Glossary QA

This teaching demo answers learner questions only from a pinned study-group glossary. It uses one real Google ADK agent, two ordinary deterministic Python tools, and one manually started ADK server shared by this Notebook and ADK Web.

## 1. Pinned study-group glossary

The repository commits the exact public JSON bytes plus a small provenance record. We verify the checksum before parsing and show metadata here without printing all glossary content.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "spooky").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this Notebook from inside the Search Agent Lab repository.")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

snapshot_path = REPOSITORY_ROOT / "spooky/data/glossary.snapshot.json"
meta_path = REPOSITORY_ROOT / "spooky/data/glossary.snapshot.meta.json"
raw_snapshot = snapshot_path.read_bytes()
metadata = json.loads(meta_path.read_text(encoding="utf-8"))
glossary_index = json.loads(raw_snapshot)

assert hashlib.sha256(raw_snapshot).hexdigest() == metadata["sha256"]
assert glossary_index["schema_version"] == metadata["schema_version"]
assert len(glossary_index["terms"]) == metadata["term_count"]

print(f"Source: {metadata['source_url']}")
print(f"Commit: {metadata['source_commit']}")
print(f"SHA-256: {metadata['sha256']}")
print(f"Schema version: {metadata['schema_version']}")
print(f"Term count: {metadata['term_count']}")
print("IDs:", [term["id"] for term in glossary_index["terms"]])

## 2. Two deterministic tools

These are ordinary Python functions. They read only the committed snapshot: one finds candidate IDs, and the other retrieves complete records with canonical glossary links.

In [ ]:
from spooky.tools import get_glossary_terms, search_glossary

search_example = search_glossary("tool and skill")
terms_example = get_glossary_terms(["tool", "skill"])

print(json.dumps(search_example, indent=2, ensure_ascii=False))
print(json.dumps(terms_example, indent=2, ensure_ascii=False))

## 3. One ADK agent

The Notebook imports the real repository agent; it does not define a second one. `temperature=0` is model configuration that reduces randomness, but it does not guarantee identical runs.

In [ ]:
from spooky import root_agent


def tool_name(tool):
    return getattr(tool, "name", getattr(tool, "__name__", type(tool).__name__))


print("Name:", root_agent.name)
print("Model:", root_agent.model)
print("Temperature:", root_agent.generate_content_config.temperature)
print("Tools:", [tool_name(tool) for tool in root_agent.tools])
print("Public instruction:")
print()
print(root_agent.instruction)

## 4. Start the shared server

From the repository root, open a VS Code terminal, run this exact command, and keep that terminal open for the complete demo:

```zsh
source .venv/bin/activate

adk api_server \
  --with_ui \
  --session_service_uri=memory:// \
  --no-reload \
  --port 8000 \
  .
```

The Notebook never starts, stops, discovers, or manages the server process.

In [ ]:
from search_agent_lab.spooky_demo import (
    ADK_WEB_URL,
    APP_NAME,
    USER_ID,
    check_server,
    create_session,
    get_session,
    run_message,
    summarize_events,
)

check_server()
print("Spooky server: ready")

## 5. Create one session

The user ID is `user`, matching the current ADK Web default. The short random suffix keeps each teaching run separate.

In [ ]:
session = create_session()
session_id = session["id"]
print("App:", APP_NAME)
print("User:", USER_ID)
print("Session:", session_id)

## 6. Run one real question

This is the canonical teaching question. The call goes through the shared server's `POST /run` endpoint—there is no Notebook Runner.

In [ ]:
QUESTION = "What is the difference between a Tool and a Skill?"
print("User question:", QUESTION)
run_events = run_message(session_id, QUESTION)
print(f"Received {len(run_events)} real ADK events.")

## 7. Inspect real events safely

The readable timeline is derived from the real returned events. It includes only allowlisted tool calls, safe glossary results, final non-thought text, safe errors, event IDs, and invocation IDs. It never dumps raw event JSON.

In [ ]:
timeline = summarize_events(run_events)
for row in timeline:
    print(json.dumps(row, ensure_ascii=False))

## 8. Verify the same session

Fetch the session through the REST API and verify that every returned event ID was persisted under the same invocation.

In [ ]:
fetched_session = get_session(session_id)
returned_event_ids = {event["id"] for event in run_events if isinstance(event.get("id"), str)}
session_events = fetched_session.get("events", [])
session_event_ids = {event["id"] for event in session_events if isinstance(event.get("id"), str)}
invocation_ids = {
    event["invocationId"]
    for event in run_events
    if isinstance(event.get("invocationId"), str) and event["invocationId"]
}
matching_session_invocations = {
    event["invocationId"]
    for event in session_events
    if event.get("id") in returned_event_ids
    and isinstance(event.get("invocationId"), str)
    and event["invocationId"]
}

assert returned_event_ids
assert returned_event_ids <= session_event_ids
assert len(invocation_ids) == 1
assert matching_session_invocations == invocation_ids
invocation_id = next(iter(invocation_ids))

print("App:", APP_NAME)
print("User:", USER_ID)
print("Session:", session_id)
print("Invocation:", invocation_id)
print("ADK Web:", ADK_WEB_URL)
print("Same-session verification: PASS")

## 9. Open ADK Web

Open [http://127.0.0.1:8000](http://127.0.0.1:8000), select `spooky`, choose the printed session, and inspect the same run in **Trace**:

- Event
- Request
- Response
- Graph

The Notebook gives a readable summary of real events. ADK Web is the local development and teaching interface for complete Request, Response, Event, and Graph inspection.

## 10. Try it yourself

Before running, predict which glossary terms and tools the agent will use. Set `RUN_TRY_IT = True` to create a separate session through the same server, then compare its safe Notebook timeline with ADK Web Trace.

Other useful questions:

- `How do a ReAct loop, an LLM Agent, and Tools work together?`
- `What is a vector database?` (the glossary should report insufficient coverage)

In [ ]:
RUN_TRY_IT = False
TRY_QUESTION = "How do a ReAct loop, an LLM Agent, and Tools work together?"

if not RUN_TRY_IT:
    print("Try-it-yourself run is off. Predict the tool flow, then set RUN_TRY_IT = True.")
else:
    check_server()
    try_session = create_session()
    try_session_id = try_session["id"]
    try_events = run_message(try_session_id, TRY_QUESTION)
    print("Question:", TRY_QUESTION)
    print("Session:", try_session_id)
    for row in summarize_events(try_events):
        print(json.dumps(row, ensure_ascii=False))